# Real tabular evaluation: a feature available too late

Before a marketing call, can a model prioritize likely subscribers? The UCI Bank Marketing export includes **call duration**, which is not available at that decision time. This notebook demonstrates an availability contract and measures how an unavailable feature changes a holdout score.

This is the notebook companion to [the full case study](../../docs/case-studies/bank-marketing.md). It uses all 41,188 records of the original additional dataset; no synthetic leakage is injected.

## Setup
From the checkout:
```bash
python -m pip install -e .
python -m pip install -r examples/requirements-notebooks.txt
python -m jupyterlab examples/notebooks
```
Select this environment's kernel and **Restart Kernel and Run All**. The loading cell explicitly downloads UCI's archive. Set `PROOFML_BANK_ARCHIVE` to an existing official outer ZIP to run offline. No API key is needed.

In [1]:
from pathlib import Path
import os
import sys
import tempfile
import pandas as pd
from IPython.display import display

# Run from the repository root or examples/notebooks, with this environment's kernel.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "examples/scifact_retrieval.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Clone proofml and launch this notebook inside that checkout.")
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
from proofml import __version__
print("ProofML", __version__)

ProofML 0.7.1


## 1. Fix the source and split

The helper verifies the archive/CSV checksums and schema. It uses the first 80% of rows for training and final 20% for evaluation, preserving UCI's published ordering. Exact timestamps and stable customer IDs are absent, so this is not a certified temporal or customer-independent split.

We do not infer event dates from month names or silently delete repeated observations.

In [2]:
from examples.bank_marketing import read_dataset, chronological_holdout, evaluate_model
archive = os.environ.get("PROOFML_BANK_ARCHIVE")
frame = read_dataset(archive=archive) if archive else read_dataset(download=True)
train, test = chronological_holdout(frame)
display(pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train), len(test)],
    "positive_rate": [train.y.eq("yes").mean(), test.y.eq("yes").mean()],
}))

,split,rows,positive_rate
0,train,32950,0.063733
1,test,8238,0.308327


## 2. Turn source knowledge into a guardrail

ProofML cannot infer business timing from a column name. Declare that duration is unavailable; require that check so an omitted declaration does not become a pass. Critical-only severity isolates this contract, not all data risks.

In [3]:
from proofml import audit, AuditPolicy
policy = AuditPolicy(severity="critical", require_checks=("feature_availability",))
undeclared = audit(train, test, target="y")
included = audit(train, test, target="y", unavailable_features=("duration",))
removed = audit(train.drop(columns="duration"), test.drop(columns="duration"),
                target="y", unavailable_features=("duration",))
reports = {"undeclared": undeclared, "duration_included": included, "duration_removed": removed}
states = {name: policy.evaluate(report).status for name, report in reports.items()}
assert states == {"undeclared": "insufficient", "duration_included": "failed", "duration_removed": "passed"}
display(pd.Series(states, name="availability_gate").to_frame())

,availability_gate
undeclared,insufficient
duration_included,failed
duration_removed,passed


Keep the **same rule after removing duration**. It will catch accidental reintroduction later. The rule checks exact predictor names, not renamed/derived post-call information. ProofML 0.7.1 fixed this workflow after the real-data experiment exposed an incorrect requirement that forbidden columns must exist.

## 3. Why an invalid model can look better

For demonstration only, fit both candidates with identical train-only preprocessing and logistic regression. A real training job should enforce the gate before fitting and not override a failed decision.

The helper uses StandardScaler + OneHotEncoder in a Pipeline, lbfgs logistic regression, C=1, max_iter=4000, seed 42, and one numerical thread. No tuning uses the test split. Unknown strings and the pdays=999 sentinel are preserved.

In [4]:
metrics = {name: evaluate_model(train, test, include_duration=include)
           for name, include in [("duration_included", True), ("duration_removed", False)]}
display(pd.DataFrame({name: {key: values[key] for key in ("roc_auc", "average_precision")}
                      for name, values in metrics.items()}).T.round(4))

,roc_auc,average_precision
duration_included,0.8211,0.5981
duration_removed,0.7483,0.5268


A higher score with duration uses information unavailable for the intended pre-call workflow. Removing it fixes one availability violation; it does not establish deployability or prove the remaining features' lineage.

These are scikit-learn metrics from one holdout/recipe, not statistical significance, prospective business performance, or measured monetary losses.

In [5]:
# Preserve other findings instead of manufacturing a clean report.
display(removed.to_frame())
broader = AuditPolicy(severity="high",
    require_checks=("feature_availability", "data_quality", "split_schema", "split_overlap"))
print("Broader review:", broader.evaluate(removed).status)

,code,severity,confidence,title,explanation,recommendation,columns,evidence
0,train_test_overlap,high,confirmed,Test observations also occur in training,"Exact normalized non-target rows match, includ...",Trace shared observations to their source and ...,(),"{'matching_test_rows': 2, 'test_rows': 8238, '..."
1,class_imbalance,medium,confirmed,Uneven class representation in train,At least one class is below the configured sha...,Report per-class metrics and a majority-class ...,"(y,)","{'split': 'train', 'classes': 2, 'minority_fra..."
2,duplicate_rows,medium,confirmed,Repeated rows in train,Repeated observations can bias estimates; thei...,Check the unit of observation before removing ...,(),"{'split': 'train', 'extra_duplicate_rows': 1637}"
3,duplicate_rows,medium,confirmed,Repeated rows in test,Repeated observations can bias estimates; thei...,Check the unit of observation before removing ...,(),"{'split': 'test', 'extra_duplicate_rows': 146}"
4,feature_distribution_shift,medium,suspicious,Distribution differs: cons.conf.idx,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(cons.conf.idx,)","{'method': 'empirical_ks_distance', 'distance'..."
5,feature_distribution_shift,medium,suspicious,Distribution differs: cons.price.idx,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(cons.price.idx,)","{'method': 'empirical_ks_distance', 'distance'..."
6,feature_distribution_shift,medium,suspicious,Distribution differs: contact,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(contact,)","{'method': 'total_variation_distance', 'distan..."
7,feature_distribution_shift,medium,suspicious,Distribution differs: emp.var.rate,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(emp.var.rate,)","{'method': 'empirical_ks_distance', 'distance'..."
8,feature_distribution_shift,medium,suspicious,Distribution differs: euribor3m,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(euribor3m,)","{'method': 'empirical_ks_distance', 'distance'..."
9,feature_distribution_shift,medium,suspicious,Distribution differs: month,Observed split distributions differ by the con...,Review sampling and population changes; examin...,"(month,)","{'method': 'total_variation_distance', 'distan..."


Broader review: failed


Matching feature rows after dropping duration need investigation, not automatic deletion: distinct customers may legitimately share the same selected attributes. Distribution shifts and unseen categories also remain. The outcome prevalence table is descriptive analysis, not a new ProofML label-shift test.

## 4. Save and enforce in a workflow

In [6]:
from proofml import AuditSuite, AuditFailed
RUN_DIR = Path(tempfile.mkdtemp(prefix="bank-notebook-", dir=ARTIFACTS))
AuditSuite(reports).save(RUN_DIR / "audits")
policy.save(RUN_DIR / "availability-policy.json")
for name, report in reports.items():
    decision = policy.evaluate(report)
    decision.save(RUN_DIR / f"{name}-decision.json")
    decision.save_junit(RUN_DIR / f"{name}-decision.xml")
try:
    policy.enforce(included)
except AuditFailed as error:
    print("Training blocked for unavailable predictor:", error.decision.status)
policy.enforce(removed)
print("Duration-removed candidate meets the availability rule, not all review requirements.")
print("Evidence:", RUN_DIR.relative_to(ROOT).as_posix())

Training blocked for unavailable predictor: failed
Duration-removed candidate meets the availability rule, not all review requirements.
Evidence: artifacts/bank-notebook-bideh7xh


## Transfer the pattern

Before training, declare post-event fields such as resolution codes, final settlement values, or outcomes recorded after a decision. Audit the **actual model inputs**, keep the declaration versioned, and require the check in CI. Availability of aggregates and publication lags still needs human review.

**Try next:** remove the declaration and observe insufficient evidence; reintroduce duration and observe failure; review a feature whose timing is ambiguous. Do not tune repeated experiments on this holdout and continue calling it untouched.

### Attribution and limits

Moro, S., Rita, P., & Cortez, P. (2014), [Bank Marketing, UCI](https://doi.org/10.24432/C5K306), CC BY 4.0 as listed by UCI. The archive's bank-additional-names.txt explains the duration restriction. Raw data is downloaded on execution and not committed here.

This tutorial does not establish fairness, temporal/customer independence, complete feature availability, or production readiness. [Detailed measured case study](../../docs/case-studies/bank-marketing.md) · [Helper code](../bank_marketing.py)